In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

In [2]:
import numpy
print(numpy.__version__)

2.0.2


In [3]:
import pkg_resources

needed = [
    "numpy", "pandas", "scikit-learn",
    "xgboost", "lightgbm", "imbalanced-learn",
    "matplotlib"
]

installed = {pkg.key: pkg.version for pkg in pkg_resources.working_set}

for name in needed:
    if name in installed:
        print(f"{name}=={installed[name]}")
    else:
        print(f"{name} (Not installed)")


C:\Users\Admin\AppData\Local\Temp\ipykernel_26932\3674952356.py:1: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


numpy==2.0.2
pandas==2.3.0
scikit-learn==1.7.1
xgboost==2.1.3
lightgbm==4.6.0
imbalanced-learn==0.14.0
matplotlib==3.10.3


In [4]:

import os, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, confusion_matrix, ConfusionMatrixDisplay

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import BorderlineSMOTE
from xgboost import XGBClassifier

# ===== 경로/설정 =====
INPUT_CSV = r"model/인사평가_feature_engineered_plus.csv"
FIXED_THRESHOLD = 0.84  # 필요 시 조정
OUTDIR = Path(INPUT_CSV).with_name("학습결과_BSMOTE")
OUTDIR.mkdir(parents=True, exist_ok=True)

# ===== 데이터 =====
if not os.path.exists(INPUT_CSV):
    raise FileNotFoundError(f"입력 CSV가 없습니다: {INPUT_CSV}")
df = pd.read_csv(INPUT_CSV)

if "업무평가" not in df.columns:
    raise KeyError("'업무평가' 열이 없습니다.")

# 타깃 매핑: 보통/0 -> 0, 좋다/좋음/1 -> 1
y = (df["업무평가"].astype(str).str.strip().str.lower()
     .map({"보통":0, "0":0, "좋다":1, "좋음":1, "1":1}).astype(int))
X = df.drop(columns=["업무평가"])

# 범주/수치 분리
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

# 전처리
numeric_tf = SimpleImputer(strategy="median")
categorical_tf = ImbPipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])
preprocess = ColumnTransformer([
    ("num", numeric_tf, num_cols),
    ("cat", categorical_tf, cat_cols),
])

# 학습/검증 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 모델
xgb = XGBClassifier(
    n_estimators=400, max_depth=6, learning_rate=0.07,
    subsample=0.9, colsample_bytree=0.9,
    objective="binary:logistic", eval_metric="logloss",
    random_state=42, n_jobs=-1, tree_method="hist"
)

# 파이프라인 (Borderline-SMOTE 포함)
model = ImbPipeline([
    ("preprocess", preprocess),
    ("bsmote", BorderlineSMOTE(random_state=42)),
    ("clf", xgb),
])

# 학습
model.fit(X_train, y_train)

# 평가 (고정 임계값)
proba = model.predict_proba(X_test)[:, 1]
pred  = (proba >= FIXED_THRESHOLD).astype(int)

acc   = accuracy_score(y_test, pred)
prec0 = precision_score(y_test, pred, pos_label=0, zero_division=0)
prec1 = precision_score(y_test, pred, pos_label=1, zero_division=0)

print("[Result]")
print(f"Used threshold : {FIXED_THRESHOLD:.2f}")
print(f"Accuracy       : {acc:.4f}")
print(f"Precision(0)   : {prec0:.4f}")
print(f"Precision(1)   : {prec1:.4f}")

# 혼동행렬 저장
cm = confusion_matrix(y_test, pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
disp.plot(values_format="d", cmap="Blues", colorbar=False)
plt.title(f"Confusion Matrix @ th={FIXED_THRESHOLD:.2f}")
plt.tight_layout()
plt.savefig(OUTDIR / "confusion_matrix.png", dpi=150)
plt.close()

print(f"\n[저장 완료] {OUTDIR}/confusion_matrix.png")


[Result]
Used threshold : 0.84
Accuracy       : 0.8537
Precision(0)   : 0.8527
Precision(1)   : 1.0000

[저장 완료] model\학습결과_BSMOTE/confusion_matrix.png


In [5]:
import joblib
from pathlib import Path

OUTDIR = Path(r"C:\Users\Admin\OneDrive\바탕 화면\인사평가ver2\학습결과_BSMOTE")
OUTDIR.mkdir(parents=True, exist_ok=True)

PKL_PATH = "model/final_model.pkl"

# model 객체 + threshold + meta 한 번에 묶어서 저장
package = {
    "model": model,             # 학습된 파이프라인
    "threshold": 0.84,          # FIXED_THRESHOLD
    "cat_cols": cat_cols,       # 범주형 칼럼
    "num_cols": num_cols        # 수치형 칼럼
}
joblib.dump(package, PKL_PATH)

print(f"[저장 완료] {PKL_PATH}")


[저장 완료] model/final_model.pkl


In [6]:
import joblib
model = joblib.load('model/final_model.pkl')
model

{'model': Pipeline(steps=[('preprocess',
                  ColumnTransformer(transformers=[('num',
                                                   SimpleImputer(strategy='median'),
                                                   ['출장', '부서', '참여프로젝트', '근속연차',
                                                    '이직회수', '주변평가', '경력',
                                                    '전년도교육출장횟수', '현회사근속년수',
                                                    '월급_KRW', '직급관리자여부',
                                                    '월급_경력비', '월급_프로젝트비',
                                                    '현근속_총경력비', '총근속', '경력성장률',
                                                    '이직빈도', '학습몰입도', '월급_근속연차비',
                                                    '월급_출장비', '근속_경력비', '평균근속기간',
                                                    '프로젝트_경력비', '프로젝트_근속연차비',
                                                    '출장_경력비', '출장_근속연차비',
                                          